In [ ]:
!pip install muspy

In [ ]:
#libraries
from music21 import corpus
import pandas as pd
import re
from pandas import Series
import numpy as np

# Search all of the files (Bach's chorales) included in the corpus
# We want the ones with 4 voices.
chorale = corpus.search(sourcePath='bach', numberOfParts=4)

#The file paths of each Chorale in Bach's file
score_series = Series(str(pth.metadata.corpusFilePath) for pth in chorale)

#we might detect identical numbers, but slight differences
#in their name, those are different harmonic interpretations
#or different file formats.
ids = score_series.str.extract(r'bwv([\d.]+)', expand=False)

#We test all chorale paths for duplicates, the paths that
#are completely the same will be removed later.
variants = score_series[ids.duplicated(keep=False)].sort_values()

variants = variants[~variants.str.endswith('.krn')]

#the chorales that we will keep. We remove the .krn files
#because the chorale already exists in .mxl form (the same chorale)
keep = ~score_series.duplicated() & (~score_series.str.endswith('.krn'))

#scores4 will contain the scores of all the 4 voices
#for all the songs in our dataset.
scores4 = [pth.parse() for pth, k in zip(chorale,keep) if k]

#we drop the duplicates | this variable exists for further testing
variants = variants.drop_duplicates()
print(variants)


In [ ]:
#We spot songs that have the same BWV id but different harmonic interpretation
#we remove the .krn files that should not exist in the dataset.
variants = variants[~variants.str.endswith(".krn")]
ids = variants.str.extract(r'bwv([\d.]+)', expand=False)
difharmon = variants[ids.duplicated(keep=False)].sort_values()
print(difharmon)

In [ ]:
#Detecting the key of a song, and the mode (major-minor)
k = scores4[0].analyze('key')
print(k, '|', k.mode, '|', k.tonic)

In [ ]:
#first measure
p = scores4[1].parts[0]
# Name of the voice
print(p.partName)
# First measure: 0 or 1; (auftakt test!)
print(p.getElementsByClass('Measure')[0].number)
# what does the first measure contain
p.measure(0).show('text')

In [ ]:
from operator import attrgetter

#Now we can start creating the dataframe, we will start by
#organising everything based on the bar, an important task
#for Roman numeral analysis.

  #First we will save the name of the attributes from music21
  #that will help us extract the data that we want and the names
  #that we want the data frame to have for those variables
measure_names_data = {"vector_name": ["bar", "suffix", "offset", "len",
                      "anacrusis"],
                        "measure_name": ["number", "numberSuffix", "offset",
                      "duration.quarterLength", "paddingLeft"]}
#the list that the data will be saved before creating the data frame
data = {"chorale_name": [],
          "bar": [],
          "suffix": [],
          "offset": [],
          "len": [],
          "anacrusis": []}

for chor_id in scores4:
  p = chor_id.parts[0]
  #First of all the measure data will be organised in order
  #to extract more data with base the time-bars, since
  #the models later will rely on time-series data.
  #One line of Measure data includes different variables
  #due to its format, those variables are not easily usable
  #and the need to be extracted later
  for m in p.getElementsByClass('Measure'):
      for mn in measure_names_data["measure_name"]:
        vector_name_index = measure_names_data["measure_name"].index(mn)
        data[measure_names_data["vector_name"][
            vector_name_index]].append(attrgetter(mn)(m))
      data["chorale_name"].append(chor_id.metadata.corpusFilePath)
#Creating the first part of the dataset
chorales = pd.DataFrame(data)
#in order to be able to handle the data later
chorales["suffix"] = chorales["suffix"].fillna('')

In [ ]:
#The current dataset
chorales

In [ ]:
from music21 import roman

#Creating a dataset with roman_scale_degrees, chord_quality, inversion,
#has7 [0, 1], certainty (music21's functionalityScore: how much that certain
#chord functions as the detected Roman numeral)
roman_dataset = {"chorale_name": [], "offset": [],
                 "roman_scale_degree": [], "chord_quality": [],
                 "inversion": [], "has7": [], "certainty": [], "beat_strength": []}

for score in scores4:
    k = score.analyze('key')
    for i in score.chordify().flatten().getElementsByClass('Chord'):
        rn = roman.romanNumeralFromChord(i, k)
        roman_dataset["chorale_name"].append(score.metadata.corpusFilePath)
        roman_dataset["offset"].append(i.offset)
        roman_dataset["roman_scale_degree"].append(rn.romanNumeralAlone)
        roman_dataset["chord_quality"].append(rn.quality)
        roman_dataset["inversion"].append(rn.inversion())
        roman_dataset["has7"].append(rn.seventh is not None)
        roman_dataset["certainty"].append(rn.functionalityScore)
        roman_dataset["beat_strength"].append(i.beatStrength)

roman_dataset = pd.DataFrame(roman_dataset)


In [ ]:
#Now we shall remove the non-roman numbers
non_roman = roman_dataset.loc[
    roman_dataset["roman_scale_degree"].isin([
        "Fr", "Ger", "It"]), "chorale_name"]

#we will remove the example that a non roman number has been detected
#It should not be more than 3 chorales and just removing a chord will
#result to noise since it will disturb the natural flow of the chords.
#So the whole chorale has to be removed.
roman_dataset = roman_dataset.loc[
    ~roman_dataset["chorale_name"].isin(non_roman)]

In [ ]:
#combining the chorales dataset (data on then the bar changes, name of the bar
#with suffix, length of each row and anacrusis)
#with the roman number dataset
roman_chord_analysis_df = pd.merge_asof(
    roman_dataset.sort_values("offset"), chorales.sort_values("offset").rename(
    columns = {"offset": "bar_start_offset"}), right_on='bar_start_offset',
    left_on='offset', by = "chorale_name")

#The dataframe was not sorted properly before, so it had to sorted
#again, first by chorale and then by offset
roman_chord_analysis_df = roman_chord_analysis_df.sort_values(by =
["chorale_name", "offset"], ascending = [True, True])

#we found earlier one chorale that two different versions of it exist in
#the dataset, let's remove it in order to prevent any data leak
roman_chord_analysis_df = roman_chord_analysis_df[
    roman_chord_analysis_df["chorale_name"] != 'bach/bwv18.5-lz.mxl']


roman_chord_analysis_df